# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library.

### Dataset Source
The dataset is described by a Croissant schema available at:

```
https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json
```

In [ ]:
# Ensure `mlcroissant` is installed in the environment
!pip install mlcroissant

## 1. Data Loading

Load the dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # Metadata is an object

print('Name:', metadata.name)
print('Description:', metadata.description)
print('Version:', getattr(metadata, 'version', ''))
print('Identifier:', getattr(metadata, 'identifier', ''))

## 2. Data Overview

Let's examine available record sets, their fields, columns, and `@id`s in the dataset.

Record sets, fields, and columns in `mlcroissant` are referenced by their `@id`. We'll list all record sets in the schema and provide details for each.

In [ ]:
# List all record sets, their @ids, and the constituent fields/columns (by @id and name)

record_sets = list(dataset.record_sets())
print(f"Found {len(record_sets)} record sets.\n")
for rs in record_sets:
    print(f"Record Set: {rs.name}")
    print(f" - @id: {rs.id}")
    if hasattr(rs, 'description'):
        print(f" - Description: {rs.description}")
    # List all fields (@id and name)
    print(" - Fields (columns):")
    for field in rs.fields:
        print(f"     * {getattr(field, 'id', field)} (name: {getattr(field, 'name', '')})")
    print()

## 3. Data Extraction

Load data from all available record sets into pandas DataFrames. We will reference record sets and fields by their `@id` as printed above.

In [ ]:
# Build a list of record set @ids
record_set_ids = [rs.id for rs in dataset.record_sets()]
dataframes = {}

for record_set_id in record_set_ids:
    print(f"Loading records for record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f" - Loaded {len(df)} records. Columns: {df.columns.tolist()}")
    print()

Let's preview data from the first available record set (by `@id`).

In [ ]:
# Use the first record set as the default for demonstration
if record_set_ids:
    example_record_set_id = record_set_ids[0]
    print(f"Example record set: {example_record_set_id}")
    print(dataframes[example_record_set_id].head())
else:
    print('No record sets found in the dataset.')

## 4. Exploratory Data Analysis (EDA)

Let's demonstrate several typical data analysis steps for a numeric field within a selected record set. We'll do so using field `@id` references as per Croissant.

In [ ]:
# Determine a numeric field for EDA

import numpy as np

# We'll auto-detect a numeric field by checking dtypes on the first DataFrame
numeric_field_id = None
group_field_id = None
df = None

if record_set_ids:
    for record_set_id in record_set_ids:
        df = dataframes[record_set_id]
        # Look for numeric column name (float or int)
        for col in df.columns:
            col_values = df[col].dropna()
            if not col_values.empty and np.issubdtype(col_values.dtype, np.number):
                numeric_field_id = col
                break
        if numeric_field_id is not None:
            # Try to find a non-numeric field for grouping
            for col in df.columns:
                if col != numeric_field_id and not np.issubdtype(df[col].dropna().dtype, np.number):
                    group_field_id = col
                    break
            break

if numeric_field_id is not None:
    threshold = df[numeric_field_id].mean() if df[numeric_field_id].notna().any() else 0
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # If we have a group field, show grouped means
    if group_field_id is not None:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped means of {numeric_field_id} by {group_field_id}:")
        print(grouped_df.head())
else:
    print('No numeric field detected for EDA in record sets.')

## 5. Visualization

Visualize the distribution of a numeric field or the relationship between two fields.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id is not None and filtered_df is not None:
    plt.figure(figsize=(8, 5))
    sns.histplot(filtered_df[numeric_field_id].dropna(), bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field_id} (filtered)")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    if group_field_id is not None:
        plt.figure(figsize=(8, 5))
        sns.boxplot(x=filtered_df[group_field_id], y=filtered_df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field_id} (filtered)")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion

- The [FAIR^2 dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa) provides mass spectrometry data for extracellular vesicles from human mast cells, enabling protein abundance and modification analysis.
- Using `mlcroissant`, we loaded the dataset metadata, explored the record sets, extracted data to pandas DataFrames, and performed basic exploratory data analysis and visualizations.
- Data field and record set references are managed using stable `@id` values from the Croissant schema for consistent, standards-compliant workflows.

You can extend this notebook for in-depth bioinformatics analysis, statistical evaluation, feature engineering, and integration with other protein or clinical datasets.